In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score,roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [2]:
#read CSV file
df=pd.read_csv('bank-additional-full.csv',sep=';')

In [3]:
#drop leakage feature
df=df.drop('duration',axis=1)

In [4]:
#Handling unknown values
all_columns=df.select_dtypes(include='object').columns
for col in all_columns:
  df=df.replace('unknown',df[col].mode()[0])

In [5]:
print(df['y'].head())

0    no
1    no
2    no
3    no
4    no
Name: y, dtype: object


In [6]:
#Label Encoding
df['y']=df['y'].map({'yes':1,'no':0})
print(df['y'].head())

0    0
1    0
2    0
3    0
4    0
Name: y, dtype: int64


In [7]:
#One Hot Encoding
df=pd.get_dummies(df,drop_first='true')

In [8]:
print(df.columns)

Index(['age', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y',
       'job_blue-collar', 'job_entrepreneur', 'job_housemaid',
       'job_management', 'job_retired', 'job_self-employed', 'job_services',
       'job_student', 'job_technician', 'job_unemployed', 'marital_divorced',
       'marital_married', 'marital_single', 'education_basic.4y',
       'education_basic.6y', 'education_basic.9y', 'education_high.school',
       'education_illiterate', 'education_professional.course',
       'education_university.degree', 'default_no', 'default_yes',
       'housing_no', 'housing_yes', 'loan_no', 'loan_yes', 'contact_telephone',
       'month_aug', 'month_dec', 'month_jul', 'month_jun', 'month_mar',
       'month_may', 'month_nov', 'month_oct', 'month_sep', 'day_of_week_mon',
       'day_of_week_thu', 'day_of_week_tue', 'day_of_week_wed',
       'poutcome_nonexistent', 'poutcome_success'],
      dtype='object')


In [9]:
#Split features and target columns
x=df.drop('y',axis=1)
y=df['y']

In [10]:
print(x.head())
print(y.head())

   age  campaign  pdays  previous  emp.var.rate  cons.price.idx  \
0   56         1    999         0           1.1          93.994   
1   57         1    999         0           1.1          93.994   
2   37         1    999         0           1.1          93.994   
3   40         1    999         0           1.1          93.994   
4   56         1    999         0           1.1          93.994   

   cons.conf.idx  euribor3m  nr.employed  job_blue-collar  ...  month_may  \
0          -36.4      4.857       5191.0            False  ...       True   
1          -36.4      4.857       5191.0            False  ...       True   
2          -36.4      4.857       5191.0            False  ...       True   
3          -36.4      4.857       5191.0            False  ...       True   
4          -36.4      4.857       5191.0            False  ...       True   

   month_nov  month_oct  month_sep  day_of_week_mon  day_of_week_thu  \
0      False      False      False             True           

In [11]:
#Train test split

X_train,X_test,y_train,y_test=train_test_split(x,y,random_state=42,stratify=y)

In [12]:
#scaling the features
scalar=StandardScaler()
X_train=scalar.fit_transform(X_train)
X_test=scalar.transform(X_test)

In [13]:
#model Training (NO SAMPLING)
model_DT=DecisionTreeClassifier(random_state=42)
model_DT.fit(X_train,y_train)
y_pred_DT=model_DT.predict(X_test)

In [15]:
#Evaluation Matrix and Confusion Matrix
cm_DT=confusion_matrix(y_test,y_pred_DT)
print(cm_DT)
print("Accurancy :",accuracy_score(y_test,y_pred_DT))
print("Precision :",precision_score(y_test,y_pred_DT))
print("Recall :",recall_score(y_test,y_pred_DT))
print("F1 :",f1_score(y_test,y_pred_DT))
print("ROC _AUC :",roc_auc_score(y_test,y_pred_DT))

[[8293  844]
 [ 753  407]]
Accurancy : 0.8449062833835098
Precision : 0.32533972821742607
Recall : 0.35086206896551725
F1 : 0.33761924512650354
ROC _AUC : 0.6292451966804165


In [16]:
#DecisionTreeCLassifier (SMOTE)
smote =SMOTE(random_state=42)
X_train_sm,y_train_sm=smote.fit_resample(X_train,y_train)

In [17]:
model_DT_sm=DecisionTreeClassifier(random_state=42)
model_DT_sm.fit(X_train_sm,y_train_sm)
y_pred_sm=model_DT_sm.predict(X_test)

In [19]:
#Evaluation Matrix and Confusion Matrix
cm_DT_sm=confusion_matrix(y_test,y_pred_sm)
print(cm_DT_sm)
print("Accurancy :",accuracy_score(y_test,y_pred_sm))
print("Precision :",precision_score(y_test,y_pred_sm))
print("Recall :",recall_score(y_test,y_pred_sm))
print("F1 :",f1_score(y_test,y_pred_sm))
print("ROC _AUC :",roc_auc_score(y_test,y_pred_sm))

[[8202  935]
 [ 727  433]]
Accurancy : 0.8385937651743226
Precision : 0.3165204678362573
Recall : 0.3732758620689655
F1 : 0.3425632911392405
ROC _AUC : 0.6354723405780967


In [26]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [40]:
scoring=['accuracy','precision','recall','f1','roc_auc']
pipeline=Pipeline([('smote_cv',SMOTE(random_state=42)),
 ('model_dt_cv',DecisionTreeClassifier(random_state=42))])
model_dt_cv=DecisionTreeClassifier(random_state=42)
results=cross_validate(pipeline,x,y,cv=skf,scoring=scoring)

In [38]:
print(results)

{'fit_time': array([0.98674273, 2.01384377, 0.86840963, 0.65048289, 0.75131559]), 'score_time': array([0.03172588, 0.05931306, 0.01689172, 0.01779509, 0.0270288 ]), 'test_accuracy': array([0.83648944, 0.82544307, 0.83539694, 0.82845696, 0.83428433]), 'test_precision': array([0.29990449, 0.27631579, 0.30685921, 0.28746713, 0.30261969]), 'test_recall': array([0.33836207, 0.33943966, 0.36637931, 0.35344828, 0.36099138]), 'test_f1': array([0.31797468, 0.30464217, 0.33398821, 0.31706138, 0.32923833]), 'test_roc_auc': array([0.62100431, 0.61462304, 0.63342515, 0.62602702, 0.62871405])}


In [42]:
print("accurancy :",results['test_accuracy'].mean())
print("precision :",results['test_precision'].mean())
print("recall :",results['test_recall'].mean())
print("f1 :",results['test_f1'].mean())
print("roc_auc :",results['test_roc_auc'].mean())

accurancy : 0.8320141476399444
precision : 0.29463326224451813
recall : 0.35172413793103446
f1 : 0.32058095472359366
roc_auc : 0.624758712077005
